# Predicting Youtube Views



In [3]:
API_KEY = "hidden"

In [44]:
#
from apiclient.discovery import build
import pprint
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib.pyplot as plt

#added for the CNNS
from PIL import Image
import requests
from io import BytesIO

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms

## 1. API Calls and Scraping Youtube metadata

The first part of this project will be the data scraping and data cleaning. We want to create a large dataset that is big enough to train an accurate machine learning model. I first got an API Key for the YoutTube Data v3 API on the google API website.

Step 1: Install the Google API client for python: % pip install --upgrade google-api-python-client

Step 2: pip install --upgrade google-auth google-auth-oauthlib google-auth-httplib2


Step 3: There are five variants of search method - Search by Keyword, Search by Location, Search Live Events, Search Related Videos and Search My videos.
-  Search by Keyword function: This will return the list of videos, channels and playlists according to the search query. By default, if the type parameter is skipped, method will display videos, channels and playlists. Default value of max results parameter is 5.

Categories: 1-28

Website I was using for reference
https://www.geeksforgeeks.org/python/youtube-data-api-for-handling-videos-set-2/




In [5]:
DEVELOPER_KEY = API_KEY
YOUTUBE_API_SERVICE_NAME = "youtube"
YOUTUBE_API_VERSION = "v3"

# creating youtube resource object
# for interacting with API
youtube = build(YOUTUBE_API_SERVICE_NAME,
                     YOUTUBE_API_VERSION,
            developerKey = DEVELOPER_KEY)


In [6]:

# def youtube_search_keyword(query, max_results):

#     # calling the search.list method to retrieve youtube search results
#     search_keyword = youtube.search().list(q = query, part = "id, snippet",
#                                                maxResults = max_results).execute()

#     # extracting the results from search response
#     results = search_keyword.get("items", [])

#     # empty list to store video, channel, playlist metadata
#     videos = []
#     playlists = []
#     channels = []

#     # extracting required info from each result object
#     for result in results:
#         # video result object
#         if result['id']['kind'] == "youtube# video":
#             videos.append("% s (% s) (% s) (% s)" % (result["snippet"]["title"],
#                             result["id"]["videoId"], result['snippet']['description'],
#                             result['snippet']['thumbnails']['default']['url']))

#         # playlist result object
#         elif result['id']['kind'] == "youtube# playlist":
#             playlists.append("% s (% s) (% s) (% s)" % (result["snippet"]["title"],
#                                  result["id"]["playlistId"],
#                                  result['snippet']['description'],
#                                  result['snippet']['thumbnails']['default']['url']))

#         # channel result object
#         elif result['id']['kind'] == "youtube# channel":
#             channels.append("% s (% s) (% s) (% s)" % (result["snippet"]["title"],
#                                    result["id"]["channelId"],
#                                    result['snippet']['description'],
#                                    result['snippet']['thumbnails']['default']['url']))

#     print("Videos:\n", "\n".join(videos), "\n")
#     print("Channels:\n", "\n".join(channels), "\n")
#     print("Playlists:\n", "\n".join(playlists), "\n")

# if __name__ == "__main__":
#     youtube_search_keyword('Popular', max_results = 5)

In [7]:
# #some of this code comes from the geekforgeeks documentation for Youtube API data usage
# def mostpopular_video_details(max_results, regionCode):
#     # Call the videos.list method to retrieve video info
#     list_videos_byid = youtube.videos().list(
#         part = "id, snippet, contentDetails, statistics",
#                   chart ='mostPopular', regionCode =regionCode,
#            maxResults = max_results, videoCategoryId =10).execute()

#     # extracting the results from search response
#     results = list_videos_byid.get("items", [])

#     # empty list to store video details
#     videos = []
#     n = 1
#     for result in results:
#         videos.append("% s (% s) (% s) (% s) (% s) (% s)"
#                         % (n, result["snippet"]["title"],
#                         result['snippet']['description'],
#                         result["snippet"]["publishedAt"],
#                                 result['contentDetails'],
#                                    result["statistics"]))
#         n = n + 1

#     print ("Videos:\n", "\n".join(videos), "\n")

# mostpopular_video_details(10, "IN")

## Creating the Dataframe

Now that we have an API call working with a function that takes in 2 arguments, region, and number of results and returns metadata for the most popular videos of that region, we can use this to create a dataframe involving data from different regions. I want to put this into a CSV file.

In [8]:
# #some of this code comes from the geekforgeeks documentation for Youtube API data usage

# # creating youtube resource object
# # for interacting with API

# #first I want to use the search, then iterate through the videos to get the video ids, and then get all of the metadata from that video id,
# #and then add that to the pandas datafram.
# #I think I should therefore create the dataframe out side of the function as a global variable

# def get_video_details(max_results, region_code,search_term):
#     #this is my search
#     search_result = youtube.search().list(part = "id,snippet", regionCode =region_code,maxResults = max_results, type="video",
#                                           q = search_term).execute()

#     video_ids = [item["id"]["videoId"] for item in search_result["items"]]

#     if not video_ids:
#         print("No videos found")
#         return pd.DataFrame()

#     videos_response = youtube.videos().list(part="id,snippet,contentDetails,statistics", id=",".join(video_ids)).execute()

#     results = videos_response.get("items", [])
#     videos = []

#     #this is me iterating through the results to get metadata for each video id

#     for result in results:
#         video = {
#         "video_id": result["id"],
#         "title": result["snippet"]["title"],
#         "published_at": result["snippet"]["publishedAt"],
#         "channel_id": result["snippet"]["channelId"],
#         "views": int(result["statistics"].get("viewCount", 0)),
#         "likes": int(result["statistics"].get("likeCount", 0)),
#         "comments": int(result["statistics"].get("commentCount", 0)),
#         "duration": result["contentDetails"]["duration"],
#         "category_id": result["snippet"]["categoryId"]
#         }
#         videos.append(video)

#     #data cleaning
#     #add differet columns such as the lengths of the titles

#     df = pd.DataFrame(videos)

#     df["title_length"] = df["title"].apply(len)
#     df["publish_hour"] = pd.to_datetime(df["published_at"]).dt.hour
#     df["publish_dayofweek"] = pd.to_datetime(df["published_at"]).dt.dayofweek
#     df["publish_month"] = pd.to_datetime(df["published_at"]).dt.month
#     df["is_weekend"] = pd.to_datetime(df["published_at"]).isin([5,6]).astype(int)

#     #create a channel column from the video
#     channel_stats = df.groupby("channel_id").agg({
#     "video_id": "count",
#     "views": "mean"
#     }).rename(columns={
#         "video_id": "channel_video_count",
#         "views": "channel_avg_views"})

#     df = df.merge(channel_stats, on="channel_id", how="left")

#     return df

In [9]:
# practice_df = get_video_details(5, "US","babyshark")

# practice_df.head(3)

In [10]:
# #Build category id dictionary so that I can look over them and search for each of them to create the database

# categories = youtube.videoCategories().list(
#     part="snippet",
#     regionCode="US").execute()

# category_map = {}

# for item in categories["items"]:
#     category_id = item["id"]
#     category_title = item["snippet"]["title"]
#     category_map[category_id] = category_title

# categories = len(category_map.values())
# print(categories)

I have a working function that scraps data using the youtube API, I want to think about the best way to create a dataset with a wide variety in terms of category and popularity. So I want to create a loop that goes through every category (1-28) and gets 100 videos from each and then merges into one bag dataframe and then turn that into a csv.

In [11]:
# dfs = [practice_df]

# for category in category_map.values():
#     print(category)

#     new_df = get_video_details(100, "US", str(category))
#     print(new_df.head(2))

#     dfs.append(new_df)

# ML_df = pd.concat(dfs, ignore_index=True)


##Training the Linear Regression Model

Now that we have a dataset we can work with. We want to use the linear regression model to predict the views based on the columns that are in the dataset. We will do some hyperparameter optimizations and cross validation to make sure we are able to get accurate prediction results

In [12]:
#load in the dataset

#df = pd.load_csv("csvfilename.csv")
#use the practice_df for now
#df = ML_df
df = pd.read_csv("Youtube.csv")
print(len(df))
#print(len(ML_df))
#df.to_csv('Youtube.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'Youtube.csv'

In [ ]:
# ML_df.head(3)

In [ ]:
#do some final datacleaning

df = df.dropna(subset=["views"])

In [ ]:
duration_seconds = []

for d in df["duration"]:
    d = d.replace("PT", "")
    hours = 0
    minutes = 0
    seconds = 0

    if "H" in d:
        hours = int(d.split("H")[0])
        d = d.split("H")[1]
    if "M" in d:
        minutes = int(d.split("M")[0])
        d = d.split("M")[1]
    if "S" in d:
        seconds = int(d.split("S")[0])

    duration_seconds.append(hours * 3600 + minutes * 60 + seconds)

df["duration_seconds"] = duration_seconds

print(df.columns.tolist())

In [ ]:
#split into x and y and training and testign data

print(df.columns.tolist())

X = df[["title_length", "publish_hour", "duration_seconds", "channel_video_count", "is_weekend", "category_id"]]
y = ["views"]

#take the log so that it is scaled for training
y = np.log1p(df["views"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [ ]:
#scale both so that we can use Ridge Linear Regression

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [ ]:
#adding cHyperparameter tuning

# Setup the pipeline steps: steps
steps = [('ridge', Ridge(fit_intercept=False))]
pipeline = Pipeline(steps)

# Specify the hyperparameter space
parameters = {'ridge__alpha': [1e-3,1e-2,1e-1,5e-1,1.0]}

# Use TimeSeriesSplit instead of the default random splits used by GridSearchCV
my_cv = TimeSeriesSplit(n_splits=2).split(X_train)

# Create the GridSearchCV object: gm_cv
gm_cv = GridSearchCV(pipeline,parameters,cv=my_cv,verbose=True,n_jobs=4,scoring='neg_mean_absolute_error',return_train_score=True)
gm_cv.fit(X_train,y_train)

# Predict
y_pred = gm_cv.predict(X_test)

means = gm_cv.cv_results_['mean_test_score']
stds  = gm_cv.cv_results_['std_test_score']
print(f"Means of CV folds: {means}")
print(f"STDs of CV folds : {stds}")

# Compute and print the metrics
print(f"Tuned Ridge Alpha: {gm_cv.best_params_}")
print(f"Tuned Ridge Score: {gm_cv.score(X_test, y_test)}")

# FIT model to tuned parameters
print('BEST FIT model')
ridge_mod = Ridge(alpha=float(gm_cv.best_params_['ridge__alpha']),fit_intercept=False)
ridge_mod.fit(X_train,y_train)
y_pred = ridge_mod.predict(X_test)

print(f"Mean Absolute Error: {mean_absolute_error(y_pred,y_test)}")
print(f"RIDGE score: {ridge_mod.score(X_test,y_test)}")
print(f"Coefficients: {ridge_mod.coef_}")

In [ ]:
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

preds = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))

print("RMSE:", rmse)

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

cv = KFold(n_splits=10, shuffle=True, random_state=342)

best_alpha = None
lowest_error = float("inf")

for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=alpha))
    ])

    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    mean_error = (-scores.mean()) ** 0.5

    if mean_error < lowest_error:
        lowest_error = mean_error
        best_alpha = alpha

print("Best alpha:", best_alpha)
print("Best CV RMSE:", lowest_error)


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

best_k = None
lowest_knn_error = float("inf")

for k in range(1, 101):

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsRegressor(n_neighbors=k))
    ])

    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    mean_error = (-scores.mean()) ** 0.5

    if mean_error < lowest_knn_error:
        lowest_knn_error = mean_error
        best_k = k

print("Best k:", best_k)
print("Best CV RMSE:", lowest_knn_error)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer

# separate numeric and text features
numeric_features = ["title_length", "publish_hour", "duration_seconds", "channel_video_count", "is_weekend", "category_id"]
text_features = "title"

# build a preprocessor that handles both
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('tfidf', TfidfVectorizer(), text_features)
])

# ridge pipeline with title text included
ridge_tfidf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('ridge', Ridge(alpha=10.0))
])

scores = cross_val_score(ridge_tfidf_pipeline, df[numeric_features + [text_features]], y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
rmse = (-scores.mean()) ** 0.5

print("Ridge + TF-IDF RMSE:", rmse)

In [ ]:
knn_tfidf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('knn', KNeighborsRegressor(n_neighbors=best_k))
])

scores = cross_val_score(knn_tfidf_pipeline, df[numeric_features + [text_features]], y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
rmse = (-scores.mean()) ** 0.5

print("KNN + TF-IDF RMSE:", rmse)

In [ ]:
!pip install vaderSentiment

In [15]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

negativity = []
positivity = []
neutrality = []

for title in df["title"]:
    score = analyzer.polarity_scores(title)
    negativity.append(score["neg"])
    positivity.append(score["pos"])
    neutrality.append(score["neu"])

df["negativity"] = negativity
df["positivity"] = positivity
df["neutrality"] = neutrality

#print(df[["title", "negativity", "positivity", "neutrality"]].head(10))

NameError: name 'df' is not defined

In [ ]:
# with sentiment features added
numeric_features_sentiment = ["title_length", "publish_hour", "duration_seconds", "channel_video_count", "is_weekend", "category_id", "negativity", "positivity", "neutrality"]
text_features = "title"

preprocessor_sentiment = ColumnTransformer([
    ('num', StandardScaler(), numeric_features_sentiment),
    ('tfidf', TfidfVectorizer(), text_features)
])

ridge_sentiment_pipeline = Pipeline([
    ('preprocessor', preprocessor_sentiment),
    ('ridge', Ridge(alpha=10.0))
])

scores = cross_val_score(ridge_sentiment_pipeline, df[numeric_features_sentiment + [text_features]], y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
rmse = (-scores.mean()) ** 0.5
print("Ridge + TF-IDF + Sentiment RMSE:", rmse)

knn_sentiment_pipeline = Pipeline([
    ('preprocessor', preprocessor_sentiment),
    ('knn', KNeighborsRegressor(n_neighbors=best_k))
])

scores = cross_val_score(knn_sentiment_pipeline, df[numeric_features_sentiment + [text_features]], y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
rmse = (-scores.mean()) ** 0.5
print("KNN + TF-IDF + Sentiment RMSE:", rmse)

In [ ]:
best_alpha_sentiment = None
lowest_sentiment_error = float("inf")

for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:

    pipeline = Pipeline([
        ('preprocessor', preprocessor_sentiment),
        ('ridge', Ridge(alpha=alpha))
    ])

    scores = cross_val_score(pipeline, df[numeric_features_sentiment + [text_features]], y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    mean_error = (-scores.mean()) ** 0.5

    if mean_error < lowest_sentiment_error:
        lowest_sentiment_error = mean_error
        best_alpha_sentiment = alpha

print("Best alpha with sentiment:", best_alpha_sentiment)
print("Ridge + TF-IDF + Sentiment CV RMSE:", lowest_sentiment_error)

In [ ]:
best_k_sentiment = None
lowest_knn_sentiment_error = float("inf")

for k in range(1, 101):

    pipeline = Pipeline([
        ('preprocessor', preprocessor_sentiment),
        ('knn', KNeighborsRegressor(n_neighbors=k))
    ])

    scores = cross_val_score(pipeline, df[numeric_features_sentiment + [text_features]], y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    mean_error = (-scores.mean()) ** 0.5

    if mean_error < lowest_knn_sentiment_error:
        lowest_knn_sentiment_error = mean_error
        best_k_sentiment = k

print("Best k with sentiment:", best_k_sentiment)
print("KNN + TF-IDF + Sentiment CV RMSE:", lowest_knn_sentiment_error)

In [16]:
kaggle_df = pd.read_csv("USvideos.csv", encoding='utf-8', on_bad_lines='skip')

# rename columns to match existing data
kaggle_df = kaggle_df.rename(columns={
    "publish_time": "published_at",
    "comment_count": "comments"
})

# add derived columns
kaggle_df["title_length"] = kaggle_df["title"].str.len()
kaggle_df["publish_hour"] = pd.to_datetime(kaggle_df["published_at"]).dt.hour
kaggle_df["publish_dayofweek"] = pd.to_datetime(kaggle_df["published_at"]).dt.dayofweek
kaggle_df["publish_month"] = pd.to_datetime(kaggle_df["published_at"]).dt.month
kaggle_df["is_weekend"] = kaggle_df["publish_dayofweek"].isin([5, 6]).astype(int)

kaggle_df = kaggle_df.dropna(subset=["views"])

print(kaggle_df.shape)
print(kaggle_df.columns.tolist())

(40949, 21)
['video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'published_at', 'tags', 'views', 'likes', 'dislikes', 'comments', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'description', 'title_length', 'publish_hour', 'publish_dayofweek', 'publish_month', 'is_weekend']


In [17]:
negativity_k = []
positivity_k = []
neutrality_k = []

for title in kaggle_df["title"]:
    score = analyzer.polarity_scores(title)
    negativity_k.append(score["neg"])
    positivity_k.append(score["pos"])
    neutrality_k.append(score["neu"])

kaggle_df["negativity"] = negativity_k
kaggle_df["positivity"] = positivity_k
kaggle_df["neutrality"] = neutrality_k

In [18]:
kaggle_y = np.log1p(kaggle_df["views"])

numeric_features_kaggle = ["title_length", "publish_hour", "is_weekend", "category_id", "negativity", "positivity", "neutrality"]
text_features = "title"

preprocessor_kaggle = ColumnTransformer([
    ('num', StandardScaler(), numeric_features_kaggle),
    ('tfidf', TfidfVectorizer(), text_features)
])

best_alpha_kaggle = None
lowest_kaggle_error = float("inf")

for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
    pipeline = Pipeline([
        ('preprocessor', preprocessor_kaggle),
        ('ridge', Ridge(alpha=alpha))
    ])
    scores = cross_val_score(pipeline, kaggle_df[numeric_features_kaggle + [text_features]], kaggle_y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    mean_error = (-scores.mean()) ** 0.5
    if mean_error < lowest_kaggle_error:
        lowest_kaggle_error = mean_error
        best_alpha_kaggle = alpha

print("Best alpha:", best_alpha_kaggle)
print("Ridge + TF-IDF + Sentiment RMSE (Kaggle):", lowest_kaggle_error)

NameError: name 'cross_val_score' is not defined

In [ ]:
best_k_kaggle = None
lowest_knn_kaggle_error = float("inf")

for k in range(1, 101):
    pipeline = Pipeline([
        ('preprocessor', preprocessor_kaggle),
        ('knn', KNeighborsRegressor(n_neighbors=k))
    ])
    scores = cross_val_score(pipeline, kaggle_df[numeric_features_kaggle + [text_features]], kaggle_y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    mean_error = (-scores.mean()) ** 0.5
    if mean_error < lowest_knn_kaggle_error:
        lowest_knn_kaggle_error = mean_error
        best_k_kaggle = k

print("Best k:", best_k_kaggle)
print("KNN + TF-IDF + Sentiment RMSE (Kaggle):", lowest_knn_kaggle_error)

Note: The original loop over k=1 to 100 was too slow on the larger Kaggle dataset (25k rows vs 1.6k rows) because KNN has to compute distances to every point, making it O(n) per prediction. We instead sample a smaller set of k values to get a reasonable estimate of the best k without the computational cost.

Took more than 30 minutes and still no results.

In [ ]:
best_k_kaggle = None
lowest_knn_kaggle_error = float("inf")

for k in [1, 3, 5, 7, 10, 15, 20, 30, 50]:
    pipeline = Pipeline([
        ('preprocessor', preprocessor_kaggle),
        ('knn', KNeighborsRegressor(n_neighbors=k))
    ])
    scores = cross_val_score(pipeline, kaggle_df[numeric_features_kaggle + [text_features]], kaggle_y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    mean_error = (-scores.mean()) ** 0.5
    print(f"k={k}, RMSE={mean_error:.4f}")

    if mean_error < lowest_knn_kaggle_error:
        lowest_knn_kaggle_error = mean_error
        best_k_kaggle = k

print("Best k:", best_k_kaggle)
print("KNN + TF-IDF + Sentiment RMSE (Kaggle):", lowest_knn_kaggle_error)

In [ ]:
#print(df.shape)
#print(df.head())

## Our Next Steps:

We want to implement cross validation and if we have time we want to be able to exxtract some data about the thumbnail and perhaps run a CNN to be able to predict which thumbnails are able to produce the most views. For this one we may need to stick to one singular youtuber (or only a couple) with videos made within roughly the same period so that there are no other external factors to this thumbnail popularity. We also may want to import a python library that is able to detect the negativity of a certain string of words so that we can add a column called negativity to our features. In the same vain, we can look at other features of the title besides it's length. One struggle we have run into that I did not anticipate is the API quota that I hit whenever I try to debug my functions, I should keep this more in mind moving foward as I hit it a couple times which set me back in terms of time for this project

Social Impact for this Progress Report:



# CNNS




In [102]:

thumbnails = kaggle_df["thumbnail_link"]
kaggle_df.head()

,video_id,trending_date,title,channel_title,category_id,published_at,tags,views,likes,dislikes,...,description,title_length,publish_hour,publish_dayofweek,publish_month,is_weekend,negativity,positivity,neutrality,subscribers
0,2kyS6SvSYSE,17.14.11,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13T17:13:01.000Z,SHANtell martin,748374,57527,2966,...,SHANTELL'S CHANNEL - https://www.youtube.com/s...,34,17,0,11,0,0.000,0.178,0.822,12700000
1,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13T07:30:00.000Z,"last week tonight trump presidency|""last week ...",2418783,97185,6146,...,"One year after the presidential election, John...",62,7,0,11,0,0.000,0.000,1.000,10100000
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146033,5339,...,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...,53,19,6,11,1,0.308,0.000,0.692,7270000
3,puqaWrEC7tY,17.14.11,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,2017-11-13T11:00:04.000Z,"rhett and link|""gmm""|""good mythical morning""|""...",343168,10172,666,...,Today we find out if Link is a Nickelback amat...,32,11,0,11,0,0.437,0.000,0.563,19600000
4,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095731,132235,1989,...,I know it's been a while since we did this sho...,24,18,6,11,1,0.000,0.000,1.000,20700000


In [80]:
import time

def get_subscriber_count_simple(channel_name, api_key):
    search_url = "https://www.googleapis.com/youtube/v3/search"
    search_params = {
        'part': 'snippet',
        'q': channel_name,
        'type': 'channel',
        'maxResults': 1,
        'key': api_key}

    search_response = requests.get(search_url, params=search_params)
    search_data = search_response.json()

    if 'items' not in search_data or len(search_data['items']) == 0:
        print(f"Channel not found: {channel_name}")
        return None

    #get the channel id and then use that channel id to get the sub count
    channel_id = search_data['items'][0]['id']['channelId']
    channel_url = "https://www.googleapis.com/youtube/v3/channels"
    channel_params = {
        'part': 'statistics',
        'id': channel_id,
        'key': api_key
    }

    channel_response = requests.get(channel_url, params=channel_params)
    channel_data = channel_response.json()

    if 'items' not in channel_data or len(channel_data['items']) == 0:
        return None

    subscriber_count = int(channel_data['items'][0]['statistics']['subscriberCount'])
    return subscriber_count

def get_all_subscriber_counts(channel_names, api_key, delay=0.5):
    subscriber_counts = []

    for i, channel_name in enumerate(channel_names):
        if i % 10 == 0:
            print(f"Fetching subscriber counts... {i}/{len(channel_names)}")

        count = get_subscriber_count(channel_name, api_key)
        subscriber_counts.append(count if count is not None else 0)

        # Add delay to avoid hitting API rate limits
        time.sleep(delay)

    return subscriber_counts

API_KEY = 'Hidden'

if 'subscribers' not in kaggle_df.columns and 'channel_title' in kaggle_df.columns:

    # unique channel names to avoid duplicate API calls
    unique_channels = kaggle_df['channel_title'].unique()
    channel_to_subs = {}
    for i, channel_name in enumerate(unique_channels):
        subs = get_subscriber_count_simple(channel_name, API_KEY)
        channel_to_subs[channel_name] = subs if subs is not None else 0
        time.sleep(0.5)

    # Map subscriber counts back to dataframe
    kaggle_df['subscribers'] = kaggle_df['channel_title'].map(channel_to_subs)
else:
    print("Available columns:", kaggle_df.columns.tolist())

Fetching subscriber counts from YouTube API...
Number of unique channels: 2207
Progress: 0/2207
Progress: 10/2207
Progress: 20/2207
Progress: 30/2207
Progress: 40/2207
Progress: 50/2207
Channel not found: JimBrowski 96HourNews
Progress: 60/2207
Progress: 70/2207
Progress: 80/2207
Progress: 90/2207
Channel not found: IISuperwomanII
Channel not found: Hopeless Records
Progress: 100/2207
Channel not found: BBC Music
Channel not found: Late Night with Seth Meyers
Channel not found: H&M
Channel not found: CamilaCabelloVEVO
Channel not found: Liza Koshy
Channel not found: TheEllenShow
Channel not found: clothesencounters
Channel not found: FoxSearchlight
Channel not found: LukeBryanVEVO
Progress: 110/2207
Channel not found: NiallHoranVEVO
Channel not found: empireofthesunvevo
Channel not found: NFVEVO
Channel not found: Marques Brownlee
Channel not found: MsAaliyahJay
Channel not found: Danny Sapko
Channel not found: Vogue
Channel not found: Michelle Khare
Channel not found: Full Frontal wit

In [81]:
kaggle_df.to_csv('Youtube(wsubs).csv', index=False)

In [96]:
kaggle_df = pd.read_csv('Youtube(wsubs).csv')
kaggle_df.shape

(40949, 25)

In [99]:
#get rid of all the channels with subscribers I couldnt get
kaggle_df = kaggle_df[kaggle_df['subscribers'] != 0]
kaggle_df.shape

(5406, 25)

In [103]:
#adding more features
X_additional = kaggle_df[numeric_features_kaggle]
scaler = StandardScaler()
X_additional = scaler.fit_transform(X_additional)

In [104]:
#some helper functions to get use the actual image from the URL

def download_image(url, timeout=10):
    response = requests.get(url, timeout=timeout)
    img = Image.open(BytesIO(response.content))
    return img

def preprocess_image(img, target_size=(64, 64)):
    if img is None:
        return np.zeros((3, *target_size))
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size)

    img_array = np.array(img)
    img_array = img_array.transpose(2, 0, 1)
    img_array = img_array / 255.0

    return img_array

 #load the images in batches
def load_images_batch(urls, target_size=(224, 224), batch_size=100):
    images = []

    for i, url in enumerate(urls):
        if i % batch_size == 0:
            print(f"Processing images {i}/{len(urls)}...")

        img = download_image(url)
        img_array = preprocess_image(img, target_size)
        images.append(img_array)

    return np.array(images)

Image_size = (64, 64)
thumbnails = thumbnails[:10000]
X_images = load_images_batch(thumbnails, target_size=Image_size)
print(f"Loaded images shape: {X_images.shape}")

#I think we should normalize by the subscriber count so that it is view per subscriber

#y = (kaggle_df['views'] / kaggle_df['subscribers']).values
#y = np.log1p(y).astype(np.float32)


Processing images 0/5406...
Processing images 100/5406...
Processing images 200/5406...
Processing images 300/5406...
Processing images 400/5406...
Processing images 500/5406...
Processing images 600/5406...
Processing images 700/5406...
Processing images 800/5406...
Processing images 900/5406...
Processing images 1000/5406...
Processing images 1100/5406...
Processing images 1200/5406...
Processing images 1300/5406...
Processing images 1400/5406...
Processing images 1500/5406...
Processing images 1600/5406...
Processing images 1700/5406...
Processing images 1800/5406...
Processing images 1900/5406...
Processing images 2000/5406...
Processing images 2100/5406...
Processing images 2200/5406...
Processing images 2300/5406...
Processing images 2400/5406...
Processing images 2500/5406...
Processing images 2600/5406...
Processing images 2700/5406...
Processing images 2800/5406...
Processing images 2900/5406...
Processing images 3000/5406...
Processing images 3100/5406...
Processing images 32

In [107]:
y = kaggle_df['views'].values / kaggle_df['subscribers'].values
y = np.log1p(y).astype(np.float32)
len(y)

5406

In [108]:
from sklearn.model_selection import train_test_split as tts

X_train, X_test, y_train, y_test = train_test_split(X_images, y, test_size=test_size, random_state=random_state)

indices = np.arange(len(X_images))
train_idx, test_idx = tts(indices, test_size=test_size, random_state=random_state)
X_add_train = X_additional[train_idx]
X_add_test = X_additional[test_idx]

# Convert to tensors
X_add_train_tensor = torch.FloatTensor(X_add_train)
X_add_test_tensor = torch.FloatTensor(X_add_test)

#convert them to tensors for pytorch
X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

#trying to combine the two datasets
train_dataset = TensorDataset(X_train_tensor, X_add_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, X_add_test_tensor, y_test_tensor)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")


Number of training batches: 136
Number of test batches: 34


In [63]:
# class CNNet(nn.Module):
#     def __init__(self):
#         super(CNNet, self).__init__()
#         self.pool1 = nn.MaxPool2d(2, 2)
#         self.relu1 = nn.ReLU() # relu
#         self.conv1 = nn.Conv2d(3, 6, 5) # takes 3-channel images

#         self.pool2 = nn.MaxPool2d(2, 2)
#         self.relu2 = nn.ReLU() # relu
#         self.conv2 = nn.Conv2d(6, 16, 5)

#         self.relu3 = nn.ReLU() # relu
#         self.fc1 = nn.Linear(16 * 5 * 5, 120)

#         self.relu4 = nn.ReLU() # relu
#         self.fc2 = nn.Linear(120, 84)

#         self.fc3 = nn.Linear(84, 10)

#     def forward(self, x):
#         x = self.pool1(F.relu(self.conv1(x)))
#         x = self.pool2(F.relu(self.conv2(x)))
#         x = x.view(-1, 16 * 5 * 5)
#         x = F.relu(self.fc1(x))
#         x = F.relu(self.fc2(x))
#         x = self.fc3(x)
#         return x

# cnn = CNNet()
# print(cnn)

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 16, 5)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv2d(16, 32, 5)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.relu2 = nn.ReLU()

        self.conv3 = nn.Conv2d(32, 64, 3)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.relu3 = nn.ReLU()

        self.fc1 = nn.Linear(64 * 5 * 5, 256)
        self.relu4 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.5)

        self.fc2 = nn.Linear(256, 128)
        self.relu5 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.relu3(x)
        x = self.pool3(x)

        #flatten it
        x = x.view(-1, 64 * 5 * 5)

        x = self.fc1(x)
        x = self.relu4(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.relu5(x)
        x = self.dropout2(x)

        x = self.fc3(x)
        return x

# Create model instance
cnn = CNN()
print(cnn)

CNN(
  (conv1): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu1): ReLU()
  (conv2): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu2): ReLU()
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu3): ReLU()
  (fc1): Linear(in_features=1600, out_features=256, bias=True)
  (relu4): ReLU()
  (dropout1): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (relu5): ReLU()
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=128, out_features=1, bias=True)
)


In [113]:
class CombinedModel(nn.Module):
    def __init__(self, num_additional_features):
        super(CombinedModel, self).__init__()

        # CNN for the thumbnail
        self.cnn = CNN()
        # Remove the final layer from CNN
        self.cnn.fc4 = nn.Identity()

        # Additional features network
        self.add_fc1 = nn.Linear(num_additional_features, 64)
        self.add_bn1 = nn.BatchNorm1d(64)
        self.add_relu1 = nn.ReLU()
        self.add_dropout1 = nn.Dropout(0.3)

        self.add_fc2 = nn.Linear(64, 32)
        self.add_relu2 = nn.ReLU()

        # Combined layers
        self.combined_fc1 = nn.Linear(128 + 32, 64)
        self.combined_bn1 = nn.BatchNorm1d(64)
        self.combined_relu1 = nn.ReLU()
        self.combined_dropout1 = nn.Dropout(0.3)

        self.combined_fc2 = nn.Linear(64, 1)

    def forward(self, image, additional):
        img_features = self.cnn(image)

        # Process additional features
        add_features = self.add_fc1(additional)
        add_features = self.add_bn1(add_features)
        add_features = self.add_relu1(add_features)
        add_features = self.add_dropout1(add_features)

        add_features = self.add_fc2(add_features)
        add_features = self.add_relu2(add_features)

        # Combine
        combined = torch.cat([img_features, add_features], dim=1)

        combined = self.combined_fc1(combined)
        combined = self.combined_bn1(combined)
        combined = self.combined_relu1(combined)
        combined = self.combined_dropout1(combined)

        output = self.combined_fc2(combined)

        return output


<class '__main__.CombinedModel'>


In [65]:
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(cnn.parameters(), lr=0.001, momentum=0.9)

criterion = nn.MSELoss()
optimizer = optim.Adam(cnn.parameters(), lr=0.001)


In [38]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5)

In [34]:
# for epoch in range(5):    # loop over dataset multiple times (epochs)

#   running_loss = 0.0
#   for i, data in enumerate(trainloader, 0):
#     # get the inputs; data is a list of [inputs, labels]
#     inputs, labels = data

#     # zero the parameter gradients
#     optimizer.zero_grad()

#     # forward + backward + optimize
#     outputs = cnn(inputs)
#     loss = criterion(outputs, labels)
#     loss.backward()
#     optimizer.step()

#     # print statistics
#     running_loss += loss.item()
#     if i % 2000 == 1999:   # print every 2000 mini-batches
#       print('[%d, %5d] loss: %.3f' %
#             (epoch + 1, i+1, running_loss/2000))
#       running_loss = 0.0

# print('Finished Training')

# # save our trained model
# PATH = './cifar_cnn.pth'
# torch.save(cnn.state_dict(), PATH)

In [109]:
def train_epoch(model, train_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    running_mae = 0.0

    for i, (images, additonal, labels) in enumerate(train_loader):
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        mae = torch.mean(torch.abs(outputs - labels))
        running_mae += mae.item()

    epoch_loss = running_loss / len(train_loader)
    epoch_mae = running_mae / len(train_loader)

    return epoch_loss, epoch_mae

def evaluate(model, test_loader, criterion):
    model.eval()
    running_loss = 0.0
    running_mae = 0.0

    with torch.no_grad():
        for images, additional, labels in test_loader:

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            mae = torch.mean(torch.abs(outputs - labels))
            running_mae += mae.item()

    epoch_loss = running_loss / len(test_loader)
    epoch_mae = running_mae / len(test_loader)

    return epoch_loss, epoch_mae

# Training parameters
EPOCHS = 50
best_val_loss = float('inf')
patience = 10
patience_counter = 0

train_losses = []
train_maes = []
val_losses = []
val_maes = []

print("Starting training...")
for epoch in range(EPOCHS):
    # Train
    train_loss, train_mae = train_epoch(cnn, train_loader, criterion, optimizer)

    val_loss, val_mae = evaluate(cnn, test_loader, criterion)

    train_losses.append(train_loss)
    train_maes.append(train_mae)
    val_losses.append(val_loss)
    val_maes.append(val_mae)

    # Learning rate scheduling
    scheduler.step(val_loss)

    # Print progress
    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f}, Train MAE: {train_mae:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val MAE: {val_mae:.4f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        torch.save(cnn.state_dict(), 'best_thumbnail_model.pth')
        print(f" New best model saved (val_loss: {val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break

print("\nTraining completed!")

# Load best model
cnn.load_state_dict(torch.load('best_thumbnail_model.pth'))
print("Best model loaded for evaluation")

Starting training...
Epoch [1/50]
  Train Loss: 3.2423, Train MAE: 0.7261
  Val Loss: 1.3169, Val MAE: 0.5054
  >>> New best model saved! (val_loss: 1.3169)
Epoch [2/50]
  Train Loss: 1.4222, Train MAE: 0.5065
  Val Loss: 1.2652, Val MAE: 0.4753
  >>> New best model saved! (val_loss: 1.2652)
Epoch [3/50]
  Train Loss: 1.2685, Train MAE: 0.4825
  Val Loss: 1.1078, Val MAE: 0.4413
  >>> New best model saved! (val_loss: 1.1078)
Epoch [4/50]
  Train Loss: 1.0543, Train MAE: 0.4335
  Val Loss: 0.9213, Val MAE: 0.3501
  >>> New best model saved! (val_loss: 0.9213)
Epoch [5/50]
  Train Loss: 0.9797, Train MAE: 0.4056
  Val Loss: 0.8618, Val MAE: 0.3473
  >>> New best model saved! (val_loss: 0.8618)
Epoch [6/50]
  Train Loss: 0.8411, Train MAE: 0.3945
  Val Loss: 0.7134, Val MAE: 0.3481
  >>> New best model saved! (val_loss: 0.7134)
Epoch [7/50]
  Train Loss: 0.7850, Train MAE: 0.3750
  Val Loss: 0.6933, Val MAE: 0.3384
  >>> New best model saved! (val_loss: 0.6933)
Epoch [8/50]
  Train Loss: 

In [105]:
kaggle_df.columns

Index(['video_id', 'trending_date', 'title', 'channel_title', 'category_id',
       'published_at', 'tags', 'views', 'likes', 'dislikes', 'comments',
       'thumbnail_link', 'comments_disabled', 'ratings_disabled',
       'video_error_or_removed', 'description', 'title_length', 'publish_hour',
       'publish_dayofweek', 'publish_month', 'is_weekend', 'negativity',
       'positivity', 'neutrality', 'subscribers'],
      dtype='object')

In [112]:
def predict_views_from_thumbnail(thumbnail_url, model, img_size=(64, 64)):
    img = download_image(thumbnail_url)
    img_array = preprocess_image(img, target_size=img_size)
    img_tensor = torch.FloatTensor(img_array).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        prediction = model(img_tensor)

    # Convert from log space
    predicted_views = np.expm1(prediction.cpu().numpy()[0][0])

    return predicted_views

example_url = "https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg"
predicted = predict_views_from_thumbnail(example_url, cnn, Image_size)
print(kaggle_df[kaggle_df['thumbnail_link'] == "https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg"]["views"])
print(f"\nPredicted views {predicted:,.0f}")

0        748374
217     2188590
448     2325233
689     2400741
924     2468267
1159    2524854
1383    2564903
Name: views, dtype: int64

Predicted views 0


In [ ]:
#doesnt perform too well so I'm gonna try and normalize with the subscriber count and also increase the number of training samples I use

In [111]:
cnn.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, addutional, labels in test_loader:
        outputs = cnn(images)
        all_predictions.append(outputs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

# Concatenate all batches
y_pred = np.concatenate(all_predictions).flatten()
y_true = np.concatenate(all_labels).flatten()

# Calculate metrics on log scale
mse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
rmse = np.sqrt(mse)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R² Score: {r2:.4f}")

Mean Squared Error (MSE): 0.4587
Root Mean Squared Error (RMSE): 0.6772
Mean Absolute Error (MAE): 0.2798
R² Score: 0.6535
